# 연도별 경기 수 세기

Drive 의 기록지 폴더(`{뿌리}/{연도}/{월}/{연도}_{월}_{팀}.xlsx`)를 훑어 **연도별 경기 수와 총 경기 수**를 센다.

- 경기 수 = **서로 다른 `game_id` 의 개수**다. 한 경기가 여러 행·여러 파일에 흩어져 있으므로 행 수를 세면 안 된다.
- 경로는 자동으로 찾는다. Colab 이면 마운트된 Drive, 이 PC 면 `local_mirror` 를 쓴다.
  단순히 폴더 존재 여부로 고르지 않는다 — 윈도우에서 `/content/...` 는 `E:\content\...` 로 해석되고, 이 PC 에는 예전에 Colab 코드를 로컬에서 돌리다 만들어진 **빈** `E:\content\drive\MyDrive-1\...` 가 남아 있어서 그걸 잡으면 0경기가 나온다. 그래서 기록지 파일이 실제로 들어 있는지까지 확인하고 고른다.
- 헤더가 깨진 파일(`2025_03_x.xlsx` 은 `game_id` 열 이름이 공백 한 칸이다)도 값 모양으로 열을 찾아 세고, 무엇을 그렇게 셌는지 마지막에 알려준다. 그냥 건너뛰면 경기가 조용히 빠진다.
- 다른 경로를 보고 싶으면 `count_games(root="...")`, 연도를 좁히려면 `count_games(years=[2024, 2025])`.

In [ ]:
import os
import re
import time
import unicodedata

import pandas as pd

# 기록지 PBP 파일 이름 규칙: 2026_06_b.xlsx (연도_월_팀머리글자)
PBP_PATTERN = re.compile(r'^\d{4}_\d{2}_[a-z]+\.xlsx$')
# game_id 로 보이는 값: 22023, 38534 처럼 네~여섯 자리. 엑셀에서 실수로 읽히면 '38534.0'.
GID_PATTERN = re.compile(r'^\d{4,6}(\.0)?$')

YEARS = range(2020, 2027)

# 위에서부터 훑어서 '기록지 파일이 실제로 들어 있는' 첫 경로를 쓴다.
CANDIDATE_ROOTS = [
    "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)",   # Colab (Drive 마운트)
    "local_mirror",                                            # 이 PC 의 Drive 사본
    "../local_mirror",
]


def has_pbp(path):
    """그 폴더 아래에 기록지 PBP 파일이 하나라도 있는가 (첫 개를 찾으면 바로 멈춘다)."""
    for _dirpath, _dirs, files in os.walk(path):
        for f in files:
            if PBP_PATTERN.match(unicodedata.normalize('NFC', f)):
                return True
    return False


def find_root(candidates=CANDIDATE_ROOTS):
    r"""시즌 폴더들이 들어 있는 뿌리 경로를 찾는다.

    '폴더가 있는가'만 보면 안 된다. 윈도우에서는 `/content/...` 가 현재 드라이브의
    `E:\content\...` 로 해석되는데, 이 PC 에는 예전에 Colab 용 코드를 로컬에서 돌리다
    만들어진 빈 `E:\content\drive\MyDrive\4-1\...` 폴더가 실제로 남아 있다. 그것만
    보고 고르면 0경기가 나온다. 그래서 기록지 파일이 들어 있는지까지 확인한다.
    """
    for c in candidates:
        if os.path.isdir(c) and has_pbp(c):
            return c
    raise FileNotFoundError(
        "기록지 폴더를 찾지 못했습니다. Colab 이면 Drive 를 먼저 마운트하고, "
        f"아니면 root= 로 경로를 직접 주세요. 찾아본 곳: {candidates}")


def _guess_gid_column(df):
    """헤더가 깨진 파일에서 game_id 열을 찾아본다. 못 찾으면 None.

    2025_03_x.xlsx 처럼 첫 열 이름이 'game_id' 가 아니라 공백 한 칸으로 저장된
    파일이 있다. 그냥 건너뛰면 그 파일의 경기가 조용히 빠져 합계가 어긋난다.
    """
    for col in df.columns:
        vals = df[col].dropna().astype(str).str.strip()
        if len(vals) == 0:
            continue
        if (vals.str.match(GID_PATTERN).mean() > 0.9) and vals.nunique() <= max(4, len(vals) // 20):
            return col
    return None


def file_game_ids(fpath):
    """한 파일의 game_id 집합과 참고 메모를 돌려준다."""
    try:
        df = pd.read_excel(fpath, usecols=["game_id"], dtype=str, engine="openpyxl")
        col, note = "game_id", ""
    except ValueError:
        df = pd.read_excel(fpath, dtype=str, engine="openpyxl")
        col = _guess_gid_column(df)
        if col is None:
            raise
        note = f"game_id 열 이름이 {col!r} 로 깨져 있음 — 값으로 찾아서 셈"

    ids = {v[:-2] if v.endswith(".0") else v
           for v in df[col].dropna().astype(str).str.strip() if v}
    return ids, note


def count_games(root=None, years=YEARS, verbose=True):
    """연도별 경기 수(= 서로 다른 game_id 수)를 센다.

    한 경기의 기록은 여러 행으로 흩어져 있고 파일도 월·팀 단위로 갈려 있으므로,
    행 수가 아니라 **game_id 의 고유 개수**가 경기 수다. 같은 game_id 가 두 파일에
    걸쳐 나올 수 있어 시즌 단위로 집합을 만들어 센다.

    돌려주는 값: (연도별 표 DataFrame, 총 경기 수, {연도: game_id 집합})
    """
    root = root or find_root()
    if verbose:
        print(f"기준 경로: {root}\n")

    rows, ids_by_year, notes = [], {}, []
    for y in years:
        base = os.path.join(root, str(y))
        if not os.path.isdir(base):
            rows.append({"연도": y, "파일": 0, "경기": 0, "비고": "폴더 없음"})
            if verbose:
                print(f"  {y}: 폴더 없음 ({base})")
            continue

        t0 = time.time()
        ids, n_files, bad = set(), 0, []
        for dirpath, _dirs, files in os.walk(base):
            for fname in sorted(files):
                if not PBP_PATTERN.match(unicodedata.normalize('NFC', fname)):
                    continue
                fpath = os.path.join(dirpath, fname)
                try:
                    got, note = file_game_ids(fpath)
                except Exception as e:
                    bad.append((fname, str(e)[:70]))
                    continue
                if note:
                    notes.append(f"{fname}: {note}")
                ids |= got
                n_files += 1

        ids_by_year[y] = ids
        rows.append({"연도": y, "파일": n_files, "경기": len(ids),
                     "비고": f"읽기 실패 {len(bad)}개" if bad else ""})
        if verbose:
            print(f"  {y}: {len(ids):>4}경기  (파일 {n_files}개, {time.time()-t0:.1f}초)")
            for fname, err in bad:
                print(f"       X 읽지 못함 {fname}: {err}")

    summary = pd.DataFrame(rows)

    # game_id 는 KBSA 전체에서 유일한 번호라 시즌끼리 겹치면 안 된다. 겹친다면 파일이
    # 엉뚱한 시즌 폴더에 들어가 있다는 뜻이고, 그러면 연도별 합과 총합이 달라진다.
    seen, dup = {}, {}
    for y, ids in ids_by_year.items():
        for gid in ids:
            if gid in seen:
                dup.setdefault(gid, [seen[gid]]).append(y)
            else:
                seen[gid] = y
    total = len(seen)

    if verbose:
        print(f"\n{'=' * 46}")
        print(summary.to_string(index=False))
        print(f"{'=' * 46}")
        print(f"합계 {total}경기 (연도별 합 {int(summary['경기'].sum())}, "
              f"파일 {int(summary['파일'].sum())}개)")
        for n in notes:
            print(f"  ! {n}")
        if dup:
            print(f"\n! 두 시즌에 걸쳐 나온 game_id {len(dup)}개 — 폴더 배치를 확인하세요")
            for gid, ys in list(dup.items())[:10]:
                print(f"    {gid}: {ys}")

    return summary, total, ids_by_year


summary, total, ids_by_year = count_games()

## (선택) full_tracked 와 대조

위는 **크롤한 기록지 원본**을 센 것이고, 아래는 검수까지 끝난 **`full_tracked_{연도}`** 에 실린 경기 수다.
두 숫자가 갈리면 아직 검수 파이프라인(노트북 2→3)을 안 돌렸거나, 로컬 사본이 Drive 보다 뒤처진 것이다.
이 셀은 서비스 계정 secrets 가 있는 파이프라인 저장소(이 PC)에서만 돌고, Colab 에서는 그냥 건너뛴다.

In [ ]:
# full_tracked_{연도} 와 대조 — 검수까지 끝난 결과에 경기가 몇 개 실렸는지 본다.
# 이 셀은 파이프라인 저장소(이 PC)에서만 돈다. Colab 에는 서비스 계정 secrets 가
# 없으므로 조용히 건너뛴다.
#
# 두 숫자가 갈리는 건 정상적인 경우가 있다:
#   · 기록지 폴더 > full_tracked : 크롤은 됐는데 아직 검수 파이프라인(노트북 2→3)을 안 돌린 경기
#   · 기록지 폴더 < full_tracked : 로컬 사본(local_mirror)이 Drive 보다 뒤처진 경우
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
try:
    from _drive_common import get_drive_service, load_by_name
except (ImportError, FileNotFoundError) as e:
    print(f"건너뜀 — scripts/_drive_common 을 쓸 수 없습니다: {e}")
else:
    svc = get_drive_service()
    rows = []
    seen = set()
    for y in YEARS:
        df = load_by_name(svc, f"full_tracked_{y}.parquet")
        if df.empty:
            df = load_by_name(svc, f"full_tracked_{y}.xlsx")
        if df.empty:
            rows.append({"연도": y, "full_tracked": 0})
            continue
        ids = {str(g).strip() for g in df["game_id"].dropna()}
        seen |= ids
        rows.append({"연도": y, "full_tracked": len(ids)})
        print(f"  {y}: {len(ids)}경기")

    ft = pd.DataFrame(rows)
    if "summary" in dir():
        cmp = summary[["연도", "경기"]].rename(columns={"경기": "기록지폴더"}).merge(ft, on="연도")
        cmp["차이"] = cmp["기록지폴더"] - cmp["full_tracked"]
        print(f"\n{cmp.to_string(index=False)}")
        print(f"\n기록지 폴더 합계 {int(cmp['기록지폴더'].sum())} / "
              f"full_tracked 합계 {len(seen)}")
    else:
        print(f"\n{ft.to_string(index=False)}\n합계 {len(seen)}경기")